In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import os

ROOT.gROOT.SetBatch(True)
ROOT.gErrorIgnoreLevel = ROOT.kError  # ROOT warning 억제

# ─────────────────────────────
# 0) 입력 파일 & 출력 경로
# ─────────────────────────────

input_samples = [
    ("hist_ST_s2018.root",   "ST_s 2018"),
    ("hist_ST_s2024.root",   "ST_s 2024"),
    ("hist_ST_t2018.root",   "ST_t 2018"),
    ("hist_ST_t2024.root",   "ST_t 2024"),
    ("hist_ST_tW2018.root",  "ST_tW 2018"),
    ("hist_ST_tW2024.root",  "ST_tW 2024"),
    ("hist_TT2018.root",     "TT 2018"),
    ("hist_TT2024.root",     "TT 2024"),
]

out_dir = "compare_ST_like_overlay_norm"
os.makedirs(out_dir, exist_ok=True)

# ─────────────────────────────
# 1) 히스토그램 이름 가져오기
#    (첫 파일 기준, plots 디렉토리)
# ─────────────────────────────

def get_hist_names_from_root(root_file):
    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open file: {root_file}")

    plots_dir = f.Get("plots")
    if not plots_dir:
        raise RuntimeError(f"'plots' directory not found in {root_file}")

    names = []
    for key in plots_dir.GetListOfKeys():
        obj = key.ReadObj()
        # TH1 계열 & 1D만 사용
        if obj.InheritsFrom("TH1") and obj.GetDimension() == 1:
            names.append(obj.GetName())
    f.Close()
    return names

hist_names = get_hist_names_from_root(input_samples[0][0])
print(f"[INFO] Found {len(hist_names)} 1D histograms in 'plots/' of {input_samples[0][0]}")

# ─────────────────────────────
# 2) b / bb 짝 찾기
#    (b_, bb_ / eta_b_, eta_bb_ / phi_b_, phi_bb_ / deltaR_b_, deltaR_bb_)
# ─────────────────────────────

pair_rules = [
    ("b_",        "bb_"),
    ("eta_b_",    "eta_bb_"),
    ("phi_b_",    "phi_bb_"),
    ("deltaR_b_", "deltaR_bb_"),
]

pairs = []    # 예: [("b_pt", "bb_pt"), ("eta_b_l", "eta_bb_l"), ...]
seen  = set() # 중복 방지

for h in hist_names:
    for s_b, s_bb in pair_rules:
        if h.startswith(s_b):
            candidate = s_bb + h[len(s_b):]
            if candidate in hist_names:
                key = tuple(sorted((h, candidate)))
                if key not in seen:
                    seen.add(key)
                    pairs.append((h, candidate))

print(f"[INFO] Found {len(pairs)} (b / bb) histogram pairs:")
for p in pairs:
    print("   ", p)

# ─────────────────────────────
# 3) x축 라벨 매핑 (원하면 계속 추가)
# ─────────────────────────────

xlabel_map = {
    "b_pt":       r"$p_T(b)$ [GeV]",
    "bb_pt":      r"$p_T(bb)$ [GeV]",
    "b_eta":      r"$\eta(b)$",
    "bb_eta":     r"$\eta(bb)$",
    "lep_pt":     r"$p_T(\ell)$ [GeV]",
    "lep_eta":    r"$\eta(\ell)$",
    "top_pt":     r"$p_T(t)$ [GeV]",
    "top_eta":    r"$\eta(t)$",
    "MET":        r"$p_T^{\mathrm{miss}}$ [GeV]",
    "eta_b_l":    r"$|\eta(b)-\eta(\ell)|$",
    "eta_bb_l":   r"$|\eta(bb)-\eta(\ell)|$",
    "eta_b_t":    r"$|\eta(b)-\eta(t)|$",
    "eta_bb_t":   r"$|\eta(bb)-\eta(t)|$",
    "phi_b_t":    r"$\Delta\phi(b,t)$",
    "phi_bb_t":   r"$\Delta\phi(bb,t)$",
    "deltaR_b_l":   r"$\Delta R(b,\ell)$",
    "deltaR_bb_l":  r"$\Delta R(bb,\ell)$",
    "deltaR_t_l":   r"$\Delta R(t,\ell)$",
    "deltaR_t_b":   r"$\Delta R(t,b)$",
    "deltaR_t_bb":  r"$\Delta R(t,bb)$",
    "deltaR_b_bb":  r"$\Delta R(b,bb)$",
}

# ─────────────────────────────
# 4) 각 pair에 대해 8개 샘플 overlay + 엔트리 정규화
# ─────────────────────────────

for h_b, h_bb in pairs:
    print(f"\n▶ Pair: {h_b} vs {h_bb}")

    # ── 각 sample에서 두 히스토그램 읽기 ──
    hist_list   = []  # 각 element: (label, counts_b, counts_bb, edges)
    edges_ref   = None
    skip_pair   = False

    for fname, label in input_samples:
        if not os.path.isfile(fname):
            print(f"   ✖ File not found: {fname}, skip this sample")
            continue

        f = ROOT.TFile.Open(fname)
        if not f or f.IsZombie():
            print(f"   ✖ Cannot open {fname}, skip for this pair")
            continue

        h1 = f.Get(f"plots/{h_b}")
        h2 = f.Get(f"plots/{h_bb}")
        if not h1 or not h2:
            f.Close()
            continue

        h1c = h1.Clone(f"{h_b}_{label}_clone");  h1c.SetDirectory(0)
        h2c = h2.Clone(f"{h_bb}_{label}_clone"); h2c.SetDirectory(0)
        f.Close()

        nb = h1c.GetNbinsX()
        # binning은 두 히스토그램 동일하다고 가정
        edges = np.array(
            [h1c.GetBinLowEdge(i) for i in range(1, nb + 1)] +
            [h1c.GetBinLowEdge(nb) + h1c.GetBinWidth(nb)]
        )
        counts_b  = np.array([h1c.GetBinContent(i) for i in range(1, nb + 1)])
        counts_bb = np.array([h2c.GetBinContent(i) for i in range(1, nb + 1)])

        if edges_ref is None:
            edges_ref = edges
        else:
            if not np.allclose(edges_ref, edges):
                print(f"   ✖ Binning mismatch in {fname} for {h_b}/{h_bb}, skip this pair")
                skip_pair = True
                break

        hist_list.append((label, counts_b, counts_bb, edges))

    if skip_pair:
        continue

    if len(hist_list) == 0:
        print(f"   ✖ No samples found for {h_b}/{h_bb}, skipping")
        continue

    # ── 각 sample별로 (b+bb) 합으로 정규화 (shape 비교용) ──
    #    → sample마다 (b+bb) integral 맞춰 줌
    norm_data = []  # (label, norm_b, norm_bb, edges)

    # 우선 전체 엔트리 합들
    integrals = []
    for label, counts_b, counts_bb, edges in hist_list:
        total = counts_b.sum() + counts_bb.sum()
        integrals.append(total)

    integrals = np.array(integrals)
    positive_mask = integrals > 0
    if not np.any(positive_mask):
        print(f"   ✖ All integrals zero for {h_b}/{h_bb}, skipping")
        continue

    min_integral = np.min(integrals[positive_mask])

    for (label, counts_b, counts_bb, edges), total in zip(hist_list, integrals):
        if total <= 0:
            print(f"   ⚠ {label} has zero integral, skipped in normalization for {h_b}/{h_bb}")
            continue
        scale = float(min_integral) / float(total)
        norm_b  = counts_b  * scale
        norm_bb = counts_bb * scale
        norm_data.append((label, norm_b, norm_bb, edges))

        print(f"   {label}: total={total:.3g}, scale={scale:.3g}")

    if len(norm_data) == 0:
        print(f"   ✖ No valid samples after normalization for {h_b}/{h_bb}, skipping")
        continue

    # ── Plot: 한 캔버스에 모든 sample의 b / bb overlay ──
    plt.style.use(hep.style.CMS)
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

    # 색은 적당히 프로세스/연도별로 나누고 싶으면 여기서 label 기반으로 바꿔도 됨
    # 기본 색상 맵: ST_s, ST_t, ST_tW, TT + 2018/2024
    base_color_map = {
        "ST_s 2018":  "tab:orange",
        "ST_s 2024":  "red",
        "ST_t 2018":  "tab:blue",
        "ST_t 2024":  "royalblue",
        "ST_tW 2018": "tab:green",
        "ST_tW 2024": "limegreen",
        "TT 2018":    "tab:purple",
        "TT 2024":    "mediumorchid",
    }

    # b/ bb를 라인 스타일로 구분
    for label, norm_b, norm_bb, edges in norm_data:
        color = base_color_map.get(label, None)

        # b
        hep.histplot(
            norm_b,
            bins=edges,
            histtype="step",
            label=f"{label} (b)",
            ax=ax,
            color=color,
            linestyle="-",
        )
        # bb
        hep.histplot(
            norm_bb,
            bins=edges,
            histtype="step",
            label=f"{label} (bb)",
            ax=ax,
            color=color,
            linestyle="--",
        )

    ax.set_ylabel("Events (normalized)")

    # x 라벨: b / bb 중 아무거나 하나 기준으로 매핑
    base_name_for_xlabel = h_b  # 혹은 h_bb
    xlabel = xlabel_map.get(base_name_for_xlabel, base_name_for_xlabel)
    ax.set_xlabel(xlabel)

    hep.cms.label("Private Work", data=False, year=2024, ax=ax)

    ax.legend(
        loc="upper right",
        prop={"size": 10},
        handletextpad=0.3,
        labelspacing=0.3,
        columnspacing=0.5,
        ncol=2,
    )

    out_name = f"compare_{h_b}_vs_{h_bb}_norm.png"
    out_path = os.path.join(out_dir, out_name)
    fig.savefig(out_path)
    plt.close(fig)

print("\n✅ All overlay *normalized* b/bb-pair plots saved in", out_dir)


[INFO] Found 29 1D histograms in 'plots/' of hist_ST_s2018.root
[INFO] Found 7 (b / bb) histogram pairs:
    ('b_pt', 'bb_pt')
    ('b_eta', 'bb_eta')
    ('eta_b_l', 'eta_bb_l')
    ('eta_b_t', 'eta_bb_t')
    ('phi_b_t', 'phi_bb_t')
    ('phi_b_l', 'phi_bb_l')
    ('deltaR_b_l', 'deltaR_bb_l')

▶ Pair: b_pt vs bb_pt
   ST_s 2018: total=3.19e+06, scale=0.0577
   ST_s 2024: total=4.77e+05, scale=0.387
   ST_t 2018: total=8.31e+05, scale=0.222
   ST_t 2024: total=1.91e+05, scale=0.964
   ST_tW 2018: total=2.54e+05, scale=0.725
   ST_tW 2024: total=1.84e+05, scale=1
   TT 2018: total=9.23e+05, scale=0.2
   TT 2024: total=8.2e+05, scale=0.225

▶ Pair: b_eta vs bb_eta
   ST_s 2018: total=3.19e+06, scale=0.0578
   ST_s 2024: total=4.76e+05, scale=0.388
   ST_t 2018: total=8.29e+05, scale=0.223
   ST_t 2024: total=1.91e+05, scale=0.966
   ST_tW 2018: total=2.54e+05, scale=0.726
   ST_tW 2024: total=1.84e+05, scale=1
   TT 2018: total=9.23e+05, scale=0.2
   TT 2024: total=8.2e+05, scale=0.225